# 🎓 Fine-Tuning: Minerva — Asistente Virtual ULS

Este notebook realiza el fine-tuning de **Llama-3.2-3B-Instruct** usando **Unsloth** para crear a **Minerva**, asistente virtual de la Universidad de La Serena.

### Stack utilizado
- 🦙 **Modelo base**: `unsloth/Llama-3.2-3B-Instruct`
- ⚡ **Framework**: Unsloth (2x más rápido que HuggingFace)
- 🎯 **Técnica**: LoRA / QLoRA (4-bit quantization)
- 📦 **Trainer**: TRL `SFTTrainer`

### Archivos necesarios
- `uls_train.jsonl` — Datos de entrenamiento
- `uls_validation.jsonl` — Datos de validación

## 1. Instalación de dependencias

In [1]:
# Instalar Unsloth (versión optimizada para Colab/Jupyter con GPU)

import subprocess, sys

# Detectar versión de CUDA y torch para instalar unsloth correcto
subprocess.run([sys.executable, '-m', 'pip', 'install', 'unsloth[colab-new]', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'trl', 'peft', 'accelerate', 'bitsandbytes', '-q'])

print('✅ Instalación completada')

✅ Instalación completada


## 2. Configuración de hiperparámetros

In [2]:
# ─────────────────────────────────────────────
#  CONFIGURACIÓN PRINCIPAL — ajusta según GPU
# ─────────────────────────────────────────────

#MODEL_NAME       = "unsloth/Llama-3.2-3B-Instruct"  # Modelo base
MODEL_NAME       = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B"  # Modelo base
MAX_SEQ_LENGTH   = 2048    # Largo máximo de secuencia. Aumentar si tienes >16GB VRAM
DTYPE            = None    # None = autodetectar (bfloat16 en Ampere+, float16 en otros)
LOAD_IN_4BIT     = True    # QLoRA: cuantización 4-bit para reducir VRAM

# LoRA — parámetros de adaptación
LORA_R           = 16      # Rank de LoRA. Mayor = más capacidad, más VRAM
LORA_ALPHA       = 16      # Escala de LoRA (generalmente igual a r)
LORA_DROPOUT     = 0       # 0 = sin dropout (recomendado con Unsloth)

# Entrenamiento
TRAIN_FILE       = "uls_train.jsonl"
VALID_FILE       = "uls_validation.jsonl"
OUTPUT_DIR       = "./uls-minerva-lora"  # Directorio donde se guardan los checkpoints
MAX_STEPS        = 120     # Pasos totales. Aumentar con más datos
BATCH_SIZE       = 2       # Por GPU. Reducir a 1 si hay OOM
GRAD_ACCUM       = 4       # Gradient accumulation (batch efectivo = BATCH_SIZE * GRAD_ACCUM)
LEARNING_RATE    = 2e-4    # Tasa de aprendizaje
WARMUP_STEPS     = 10      # Pasos de warmup

print('✅ Configuración cargada')
print(f'   Modelo: {MODEL_NAME}')
print(f'   Modo:   {"QLoRA 4-bit" if LOAD_IN_4BIT else "LoRA 16-bit"}')
print(f'   LoRA r: {LORA_R} | alpha: {LORA_ALPHA}')

✅ Configuración cargada
   Modelo: unsloth/DeepSeek-R1-Distill-Qwen-1.5B
   Modo:   QLoRA 4-bit
   LoRA r: 16 | alpha: 16


## 3. Carga del modelo base

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
    # token = "hf_...",  # Descomenta si usas un modelo privado de HuggingFace
)

print('✅ Modelo base cargado')
print(f'   Parámetros totales: {sum(p.numel() for p in model.parameters()):,}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/benja/miniconda3/envs/IAmodel_v3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.622 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/deepseek-r1-distill-qwen-1.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Modelo base cargado
   Parámetros totales: 1,187,853,824


## 4. Aplicar LoRA al modelo

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # Atención
        "gate_proj", "up_proj", "down_proj",        # MLP / FFN
    ],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",      # No entrenar bias
    use_gradient_checkpointing = "unsloth",  # Ahorra ~30% VRAM
    random_state   = 42,
    use_rslora     = False,       # Rank-stabilized LoRA (experimental)
)

# Mostrar parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA aplicado')
print(f'   Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}% del total)')

Unsloth 2026.4.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA aplicado
   Parámetros entrenables: 18,464,768 (1.53% del total)


## 5. Preparación del dataset

Los datos están en formato **conversacional** (system / user / assistant). Usamos el chat template de Llama-3.

In [5]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Aplicar el chat template de Llama-3
tokenizer = get_chat_template(
    tokenizer,
    #chat_template = "llama-3.1",
    chat_template="qwen-2.5",
)

def format_conversations(examples):
    """Convierte la lista de mensajes al formato de texto del chat template."""
    texts = []
    for conversation in examples["conversations"]:
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize          = False,
            add_generation_prompt = False,
        )
        texts.append(text)
    return {"text": texts}

# Cargar datasets
train_dataset = load_dataset(
    "json",
    data_files = {"train": TRAIN_FILE},
    split      = "train",
)
valid_dataset = load_dataset(
    "json",
    data_files = {"validation": VALID_FILE},
    split      = "validation",
)

# Aplicar formato
train_dataset = train_dataset.map(format_conversations, batched=True)
valid_dataset = valid_dataset.map(format_conversations, batched=True)

print(f'✅ Dataset cargado')
print(f'   Ejemplos de entrenamiento: {len(train_dataset)}')
print(f'   Ejemplos de validación:    {len(valid_dataset)}')
print()
print('--- Primer ejemplo (preview) ---')
print(train_dataset[0]['text'][:500], '...')

✅ Dataset cargado
   Ejemplos de entrenamiento: 8
   Ejemplos de validación:    2

--- Primer ejemplo (preview) ---
<|im_start|>system
Eres una asistente virtual de la Universidad de La Serena (ULS), Chile. Tu nombre es Minerva. Tu rol es apoyar a los estudiantes respondiendo consultas sobre trámites, documentos, reglamentos y procesos académicos de la universidad. Hablas con un tono formal pero amigable, siempre dispuesta a orientar al estudiante de manera clara y precisa. Cuando no tengas información suficiente, indicas amablemente dónde puede el estudiante obtener más detalles.<|im_end|>
<|im_start|>user
H ...


## 6. Configurar el Trainer (SFTTrainer)

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = valid_dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LENGTH,
    dataset_num_proc= 2,
    packing         = False,  # True acelera entrenamiento con secuencias cortas
    args = TrainingArguments(
        per_device_train_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps  = GRAD_ACCUM,
        warmup_steps                 = WARMUP_STEPS,
        max_steps                    = MAX_STEPS,
        learning_rate                = LEARNING_RATE,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        eval_steps                   = 30,
        save_strategy                = "steps",
        save_steps                   = 60,
        optim                        = "adamw_8bit",  # Optimizer 8-bit para ahorrar VRAM
        weight_decay                 = 0.01,
        lr_scheduler_type            = "linear",
        seed                         = 42,
        output_dir                   = OUTPUT_DIR,
        report_to                    = "none",  # Cambiar a 'wandb' si usas W&B
    ),
)

print('✅ Trainer configurado')

num_proc must be <= 8. Reducing num_proc to 8 for dataset of size 8.
[datasets.arrow_dataset|WARNING]num_proc must be <= 8. Reducing num_proc to 8 for dataset of size 8.
Unsloth: Tokenizing ["text"] (num_proc=8): 100%|██████████| 8/8 [00:01<00:00,  7.31 examples/s]
num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
[datasets.arrow_dataset|WARNING]num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 2/2 [00:00<00:00,  3.90 examples/s]

✅ Trainer configurado


## 7. Entrenamiento 🚀

In [7]:
import time

# Mostrar uso de GPU antes
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU: {gpu_stats.name} | VRAM total: {max_memory} GB | Reservado: {start_gpu_memory} GB')

# Entrenar
start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time

# Mostrar estadísticas
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f'\n✅ Entrenamiento completado')
print(f'   Tiempo total:         {elapsed/60:.1f} minutos')
print(f'   VRAM usada:           {used_memory} GB / {max_memory} GB')
print(f'   Loss final:           {trainer_stats.training_loss:.4f}')

GPU: NVIDIA GeForce RTX 3060 | VRAM total: 11.622 GB | Reservado: 1.779 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 120 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Step,Training Loss
10,3.498100
20,2.507700
30,1.621400
40,0.963900
50,0.418900
60,0.115800
70,0.029700
80,0.014800
90,0.011500
100,0.010200



✅ Entrenamiento completado
   Tiempo total:         3.4 minutos
   VRAM usada:           4.418 GB / 11.622 GB
   Loss final:           0.7676


## 8. Prueba de inferencia — ¡Habla con Minerva! 💬

In [8]:
from unsloth.chat_templates import get_chat_template

# Activar modo de inferencia (2x más rápido)
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "Eres una asistente virtual de la Universidad de La Serena (ULS), Chile. "
    "Tu nombre es Minerva. Tu rol es apoyar a los estudiantes respondiendo consultas "
    "sobre trámites, documentos, reglamentos y procesos académicos de la universidad. "
    "Hablas con un tono formal pero amigable, siempre dispuesta a orientar al estudiante "
    "de manera clara y precisa."
)

def chat_minerva(user_message: str, max_new_tokens: int = 512) -> str:
    """Envía un mensaje a Minerva y devuelve su respuesta."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_message},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids        = inputs,
        max_new_tokens   = max_new_tokens,
        use_cache        = True,
        temperature      = 0.7,
        top_p            = 0.9,
    )
    # Decodificar solo los tokens nuevos
    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens = True,
    )
    return response.strip()


# ─── PRUEBAS ───
test_questions = [
    "¿Cómo solicito un certificado de alumno regular?",
    "¿Cuáles son los requisitos para un examen de repetición?",
    "Necesito información sobre el proceso de titulación",
]

for q in test_questions:
    print(f'👤 Estudiante: {q}')
    resp = chat_minerva(q)
    print(f'🎓 Minerva: {resp}')
    print('─' * 70)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


👤 Estudiante: ¿Cómo solicito un certificado de alumno regular?
🎓 Minerva: ¡Con gusto te ayudo con que! Los certificados de alumnas en la ULS están regulados por el Reglamento del Sistema, que puede ser encontrado en el Portfolio estudianto o directamente en la Lista de certificaciones de la universidad.

Para solicitar el certificado de alumno regular, debes providing:
1. La información del certificado (Nombre al que será accumulado, Nombre del alumnaro y el título de la asignatura)
2. El tono de discapacidad (en la Universidad)
3. El nombre despano (si hay dos personas a considerar)
4. Laismatic de la asignatura (si aplica el teto de reprobación)

La información de teto reprobación depende de la asignatura, que puede variar según el sistema, por lo general es 1.0 (enocto de 20/21).

El proceso consiste en:
1. Descargar el documento de reprobación desde el Portfolio estudianto.
2. Completar el formulario de solicitud en el portal estudiantil de la universidad.
3. Entregar el formulario

## 9. Guardar el modelo fine-tuneado

In [9]:
# ─── OPCIÓN A: Guardar solo los adaptadores LoRA (liviano, ~50-200 MB) ───
# Recomendado para compartir o continuar entrenamiento
model.save_pretrained("uls-minerva-lora-adapters-DS")
tokenizer.save_pretrained("uls-minerva-lora-adapters-DS")
print('✅ Adaptadores LoRA guardados en ./uls-minerva-lora-adapters')

✅ Adaptadores LoRA guardados en ./uls-minerva-lora-adapters


In [10]:
# ─── OPCIÓN B: Guardar modelo completo fusionado (16-bit) ───
# Útil para despliegue con vLLM, Ollama, etc.
model.save_pretrained_merged(
    "uls-minerva-merged-16bit-DS",
    tokenizer,
    save_method = "merged_16bit",
)
print('✅ Modelo fusionado (16-bit) guardado en ./uls-minerva-merged-16bit')

Found HuggingFace hub cache directory: /home/benja/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [05:13<00:00, 313.79s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:08<00:00,  8.27s/it]


Unsloth: Merge process complete. Saved to `/home/benja/Escritorio/prueba/uls-minerva-merged-16bit-DS`
✅ Modelo fusionado (16-bit) guardado en ./uls-minerva-merged-16bit


In [11]:
# ─── OPCIÓN C: Exportar a GGUF para usar con Ollama/LM Studio ───
# Formatos disponibles: 'q4_k_m' (recomendado), 'q8_0', 'f16'
model.save_pretrained_gguf(
    "uls-minerva-gguf-DS",
    tokenizer,
    quantization_method = "q4_k_m",
)
print('✅ Modelo GGUF guardado en ./uls-minerva-gguf')
print('   Puedes cargarlo con: ollama create minerva -f ./uls-minerva-gguf/Modelfile')

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/benja/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [05:01<00:00, 301.56s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.61s/it]


Unsloth: Merge process complete. Saved to `/home/benja/Escritorio/prueba/uls-minerva-gguf-DS`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['uls-minerva-gguf-DS_gguf/deepseek-r1-distill-qwen-1.5b.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['uls-minerva-gguf-DS_gguf/deepseek-r1-distill-qwen-1.5b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/deepseek-r1-distill-qwen-1.5b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /home/benja/.unsloth/llama.cpp/llama-cli --model uls-minerva-gguf-DS_gguf/deepseek-r1-distill-qwen-1.5b.Q4_K_M.gguf -p "why is the sky blue?"
✅ Modelo GGUF guardado en ./uls-minerva-gguf
   Puedes cargarlo con: ollama create minerva -f ./uls-minerva-gguf/Modelfile


In [12]:
# ─── OPCIÓN D: Subir a HuggingFace Hub ───
# Descomenta y reemplaza con tu token y nombre de repositorio

# HF_TOKEN = "hf_..."
# REPO_NAME = "tu-usuario/uls-minerva-lora"
# model.push_to_hub(REPO_NAME, token=HF_TOKEN)
# tokenizer.push_to_hub(REPO_NAME, token=HF_TOKEN)
# print(f'✅ Modelo subido a https://huggingface.co/{REPO_NAME}')

## 10. Consejos para mejorar el modelo

### 📊 Más datos = mejor modelo
Con solo 8 ejemplos de entrenamiento el modelo aprenderá el tono y formato, pero para un asistente real necesitas al menos **50-200 ejemplos** cubriendo:
- Más trámites y procesos específicos de la ULS
- Casos de edge (consultas ambiguas, preguntas fuera de tema)
- Variedad de formas de hacer la misma pregunta

### 🔧 Parámetros a ajustar si tienes más datos
| Parámetro | Dataset pequeño (<50) | Dataset mediano (50-500) | Dataset grande (>500) |
|-----------|----------------------|--------------------------|----------------------|
| `MAX_STEPS` | 60-120 | 200-500 | 1000+ |
| `LORA_R` | 8-16 | 16-32 | 32-64 |
| `LEARNING_RATE` | 2e-4 | 1e-4 | 5e-5 |

### 📄 Cómo agregar documentos reales de la ULS
1. Descarga los reglamentos en PDF del sitio oficial de la ULS
2. Extrae el texto con `pdfplumber` o `pymupdf`
3. Genera pares QA con un LLM (ej: Claude o GPT-4) a partir del texto
4. Revisa y corrige manualmente la calidad de las respuestas
5. Agrega los pares al JSONL de entrenamiento

### 🚀 Para producción
Considera usar **RAG (Retrieval-Augmented Generation)** junto con el fine-tuning:
el fine-tuning le enseña el tono y formato, y RAG le provee información actualizada de documentos reales.